# Kinked Consumption Functions with Preference Shocks

Consumers face a higher interest rate when borrowing than when saving
(`Rboro > Rsave`) and shocks $\eta$ to the utility of consumption. Each line is the
consumption function at one value of $\eta$. Move the slider to change the coefficient of
relative risk aversion, `CRRA`.

To open this notebook as a dashboard, with the code hidden, run
`uv run voila dashboard/REMARK-starter-example-dashboard.ipynb`, or use the Binder link in
the README.

In [1]:
import io

import numpy as np
from HARK.ConsumptionSaving.ConsPrefShockModel import KinkyPrefConsumerType
from ipywidgets import FloatSlider, Image, VBox
from matplotlib.figure import Figure

In [2]:
def draw(CRRA):
    """Solve the model at this CRRA and return its consumption functions as a PNG image."""
    agent = KinkyPrefConsumerType(CRRA=CRRA)
    agent.cycles = 0  # infinite horizon
    agent.solve()

    solution = agent.solution[0]
    m = np.linspace(solution.mNrmMin, 5, 200)
    fig = Figure(figsize=(6, 4), layout="tight")
    ax = fig.subplots()
    for eta in agent.PrefShkDstn[0].atoms[0]:
        ax.plot(m, solution.cFunc(m, eta * np.ones_like(m)))
    ax.set_xlabel("Market resources $m$")
    ax.set_ylabel(r"Consumption $c(m, \eta)$")
    ax.set_ylim(0.0, 3.5)

    png = io.BytesIO()
    fig.savefig(png, format="png")
    return png.getvalue()

In [3]:
# The slider redraws the figure in an Image widget. (ipywidgets' `interact` is shorter,
# but a notebook whose `interact` draws a large figure sometimes never finishes when run
# unattended, as Voila and nbconvert run it.)
CRRA = FloatSlider(value=2.0, min=1.0, max=5.0, step=0.05, description="CRRA")
figure = Image(value=draw(CRRA.value), format="png")
CRRA.observe(lambda change: setattr(figure, "value", draw(change["new"])), names="value")
VBox([CRRA, figure])